# LLM Programming Experiment on Google Colab

This notebook allows you to run programming experiments using Ollama (Gemma 2, Llama 3.1, Qwen 2.5) on a Google Colab T4 GPU.

**Instructions:**
1. Connect to a **T4 GPU** runtime (Runtime > Change runtime type).
2. Run the cells in order.
3. Upload your `problems.zip` when prompted.



## 1. Setup Environment and Ollama



In [ ]:
# @title Install Dependencies and Ollama
import os
import subprocess
import time
import requests
import json
import re
import zipfile
from datetime import datetime
from google.colab import files

print("Installing zstd and Ollama...")
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
print("Starting Ollama server...")
subprocess.Popen('nohup ollama serve > /tmp/ollama.log 2>&1 &', shell=True)

# Wait for server
for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            print(f'Ollama server ready after {i+1}s')
            break
    except:
        pass
    time.sleep(1)
else:
    print('ERROR: Ollama server failed to start.')



## 2. Configuration



In [ ]:
# @title Experiment Parameters
# @markdown Choose the model and the range of problems to test.

model_choice = "qwen2.5:7b" # @param ["gemma2:9b", "llama3.1:8b", "qwen2.5:7b"]
start_problem = 1 # @param {type:"integer"}
end_problem = 10 # @param {type:"integer"}

print(f"Pulling model {model_choice}...")
!ollama pull {model_choice}



## 3. Data Loading



In [ ]:
# @title Upload and Parse problems.zip
if not os.path.exists('problems.zip'):
    print("Please upload problems.zip")
    uploaded = files.upload()
else:
    print("problems.zip already exists.")

if os.path.exists('problems'):
    !rm -rf problems

with zipfile.ZipFile('problems.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

def parse_problem_file(file_path):
    """Parses a problem .txt file into a structured dictionary."""
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    test_cases = []
    tests_match = re.search(r"Test cases:\n(.*?)$", content, re.DOTALL)
    if tests_match:
        test_lines = tests_match.group(1).strip().split('\n')
        test_cases = [line.strip() for line in test_lines if line.strip().startswith('assert')]

    return {
        "task_id": os.path.basename(file_path).replace('.txt', ''),
        "description": content.strip()
    }

problems_to_test = []
for i in range(start_problem, end_problem + 1):
    filename = f"{i:04d}.txt"
    path = os.path.join('problems', filename)
    if os.path.exists(path):
        problems_to_test.append(parse_problem_file(path))
    else:
        print(f"Warning: {filename} not found in problems/ folder.")

print(f"Loaded {len(problems_to_test)} problems to test.")



## 4. Execution and Evaluation



In [ ]:
# @title Run Experiment
import sys
from io import StringIO

def call_ollama(model, system_prompt, user_prompt):
    start = time.time()
    try:
        if system_prompt:
            r = requests.post("http://localhost:11434/api/generate", json={
                "model": model, "prompt": user_prompt, "system": system_prompt,
                "stream": False, "options": {"temperature": 0.2, "num_predict": 2048}
            }, timeout=300)
        else:
            r = requests.post("http://localhost:11434/api/generate", json={
                "model": model, "prompt": user_prompt, "raw": True,
                "stream": False, "options": {"temperature": 0.2, "num_predict": 2048}
            }, timeout=300)

        r.raise_for_status()
        data = r.json()
        return data.get("response", ""), {
            "duration": time.time() - start,
            "prompt_tokens": data.get("prompt_eval_count", 0),
            "completion_tokens": data.get("eval_count", 0),
            "success": True, "error": ""
        }
    except Exception as e:
        return "", {"duration": time.time() - start, "prompt_tokens": 0,
                    "completion_tokens": 0, "success": False, "error": str(e)}

def extract_python_code(response: str) -> str:
    matches = re.findall(r"```python\s*\n(.*?)```", response, re.DOTALL)
    if matches: return matches[-1].strip()
    matches = re.findall(r"```\s*\n(.*?)```", response, re.DOTALL)
    if matches: return matches[-1].strip()
    return ""

def evaluate_code(code, test_list):
    if not code:
        return {"passed": False, "tests_passed": 0, "errors": ["No code extracted"]}

    passed_count = 0
    errors = []
    # Using exec for evaluation in Colab environment
    for test in test_list:
        local_ns = {}
        try:
            exec(code + "\n" + test, {}, local_ns)
            passed_count += 1
        except Exception as e:
            errors.append(f"Test failed: {test} | Error: {str(e)}")

    return {
        "passed": passed_count == len(test_list) and len(test_list) > 0,
        "tests_passed": passed_count,
        "tests_total": len(test_list),
        "errors": errors
    }

#SYSTEM_PROMPT = "You are an expert Python programmer. Write a Python function that solves the given task. Provide your final answer as code inside a ```python code block."
SYSTEM_PROMPT = None
results = []
output_filename = f"{model_choice.replace(':', '-')}_{start_problem}_{end_problem}.json"

print(f"Starting experiment... Output will be saved to {output_filename}")

for prob in problems_to_test:
    task_id = prob['task_id']
    print(f"Testing problem {task_id}...", end=" ")

    response_text, meta = call_ollama(model_choice, SYSTEM_PROMPT, prob['description'])

    if not meta['success']:
        print(f"API Error: {meta['error']}")
        results.append({
            "task_id": task_id,
            "model": model_choice,
            "passed": False,
            "error": meta['error']
        })
        continue

    extracted_code = extract_python_code(response_text)
    eval_result = evaluate_code(extracted_code, prob['test_list'])

    status = "PASS" if eval_result['passed'] else "FAIL"
    print(f"{status} ({eval_result['tests_passed']}/{eval_result['tests_total']} tests)")

    results.append({
        "timestamp": datetime.now().isoformat(),
        "model": model_choice,
        "task_id": task_id,
        "passed": eval_result['passed'],
        "tests_passed": eval_result['tests_passed'],
        "tests_total": eval_result['tests_total'],
        "errors": eval_result['errors'],
        "duration_seconds": meta['duration'],
        "prompt_tokens": meta['prompt_tokens'],
        "completion_tokens": meta['completion_tokens'],
        "response_text": response_text,
        "extracted_code": extracted_code
    })

# Save results
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nExperiment complete. Saved {len(results)} results to {output_filename}")
files.download(output_filename)
